# Data Preprocessing

_Demonstrates data preprocessing (cleaning and transformation) activities on suitable data set._

In [1]:
# imports required modules

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import matplotlib.pyplot as plt

## Retrieving & Analyzing the Data

In [ ]:
# Loads and displays the data

housing = [CODE HERE TO LOAD GIVEN .CSV FILE USING PANDAS read_csv METHOD]

In [ ]:
[CODE HERE TO DISPLAY THE DATA]

In [ ]:
# Print basic information about the data

[CODE HERE TO PRINT THE BASIC INFO ABOUT THE DATASET]

In [ ]:
# Looks at the statistics for 'ocean_proximity'

housing.ocean_proximity.value_counts()

In [ ]:
# Looks at the basic statistics of the data

[CODE HERE TO PRINT THE BASIC STATISTICS OF THE DATA]

_**NOTE ALL YOUR OBSERVATIONS BELOW.**_

1) ...
2) ...
3) ...

In [ ]:
# Plots the histogram of each numerical attribute to look at the data distributions.

housing.hist(bins=50, figsize=(15,8))
plt.show()

_**WRITE YOUR OBSERVATIONS ON HISTOGRAMS.**_

## Cleaning Data

**Removing duplicate observations**

In [ ]:
# Checks for duplicate observations
duplicate_count = sum(housing.duplicated())
print("Duplicate observations found in the dataset: {}".format(duplicate_count))

# Deletes the duplicate data, if found
if duplicate_count > 0:
    housing.drop_duplicates(inplace=True)
    print("\n\tDuplicate observations were deleted.")
    print("\n\tData shape after duplicate removal:", housing.shape)

**Removing single-valued columns**

In [ ]:
# Gets number of unique values for each column
unique_values_per_attrib = housing.nunique()

# Records columns to delete
single_value_columns = [i for i, value_count in enumerate(unique_values_per_attrib) if value_count == 1]
print("Number of single_valued columns found in the datsset: {}".format(len(single_value_columns)))

# Deletes single-value columns, if exist
if len(single_value_columns) > 0:
    housing.drop(single_value_columns, axis=1, inplace=True)
    print("\n\tAll single-valued columns were removed.")
    print("\n\tData shape after removal of single-value column(s):", housing.shape)

## Creating Test Set

Seperates a part of data to be used for testing trained machine learning models to evaluate their prediction performance.

In [9]:
# Option 1
# train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

# Option 2
housing["median_income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5])

In [ ]:
# Views distribution of median income categories

housing["median_income_cat"].hist()
plt.show()

In [ ]:
# Splits data into train [80%] and test set [20%] applying stratification on column median_income_cat

train_set, test_set = train_test_split([CODE HERE], random_state=42)

In [12]:
# Removes temporary attribute 'median_income_cat' from both the datasets as 
# this attribute will no longer be used later.

train_set.drop("median_income_cat", axis=1, inplace=True)
test_set.drop("median_income_cat", axis=1, inplace=True)

## Preparing Data for Machine Learning Algorithms

There are two options to create test set.
- Using random sampling method to select observerations
- Using stratified sampling method to ensure both train and test set have same distribution

Let's go with the second option.

In [13]:
# First, seperates target from features

housing_train = train_set.drop("median_house_value", axis=1)
housing_train_labels = train_set["median_house_value"].copy()

### Handling Missing Values

Though the missing values are there only in attribute $total\_bedrooms$, but all the numeric columns are made available to Imputer so that it can impute missing values in any attributes in test set.

In [ ]:
# Initialize imputer with "median" strategy
imputer = SimpleImputer([CODE HERE])

# Considers only numeric columns as the imputer works on numeric data
housing_train_num = housing_train.drop("ocean_proximity", axis=1)

# Fits the imputer
imputer.fit(housing_train_num)

# Transforms the missing values in each column with learned median
housing_train_num_imputed = imputer.transform(housing_train_num)

### Scaling Features

Applies scaling transformations to relevant attributes as machine learning algorithms don't work well with (numerical) attributes having different scales.

The options are
- Min-max scaling (also known as normalization)
- Standardization

Let's go with the second option.

In [ ]:
# Initializes the scaler
std_scaler = StandardScaler()

# Fits the scaler and then transforms numerical train set housing_train_num_imputed
housing_train_num_scaled = [CODE HERE]

In [ ]:
# Prints the shape of the transformed dataset with only numerical attributes
housing_train_num_scaled.shape

### Encoding Categorical Attributes

**Encods categorical data in attribute $ocean\_proximity$.**

Options are

1) Ordinal encoding
2) Binary encoding

Let's go with the second option.

In [ ]:
# Initializes encoder
cat_encoder = OneHotEncoder(sparse_output=False)

# Sets a variable to attribute column
housing_train_cat = housing_train[["ocean_proximity"]]

# Fits the encoder
[CODE HERE TO FIT THE ENCODER cat_encoder ON housing_train_cat]

# Transforms the categorical data by transforming it
housing_train_cat_1hot = [CODE HERE TRANSFORM housing_train_cat OVER THE FITTED cat_encoder]

In [ ]:
# Shows one-hot encoded information in densed array form [just for reference]
housing_train_cat_1hot

In [ ]:
# Shows the ordered list of categories related to one-hot encoding
cat_encoder.categories_

Now, instead of transforming numeric and non-numeric attributes individually and then integrating back, transformation pipelines that can take care of tranforming both these different types of attributes in an integrated way can be used as shown below.

### Combining Transformed Data

In [ ]:
# Stacks numerically scaled data and categorically encoded data horizontally (column wise).
housing_train_transformed = np.hstack((housing_train_num_scaled, housing_train_cat_1hot))

# The above stackign can also be done over the following statement
# housing_train_transformed = np.append(housing_train_num_scaled, housing_train_cat_1hot, axis=1)

# Checks the shape of transformed train set
housing_train_transformed.shape

In [ ]:
# Just for reference, the train set is shown through DataFrame 
pd.DataFrame(
    housing_train_transformed, 
    columns=list(housing_train_num.columns) + cat_encoder.categories_[0].tolist())

In [55]:
# Saves the transformed train set to be used for machine learning training later
# NOTE: Target also gets saved in the last column

with open("./housing_train_transformed.npy", "wb") as f:
    np.save(
        f, 
        np.concatenate((housing_train_transformed, np.expand_dims(housing_train_labels, axis=1)), axis=1), 
        allow_pickle=False)

### Transforming Test Data

In [56]:
# Seperates target from features

housing_test = test_set.drop("median_house_value", axis=1)
housing_test_labels = test_set["median_house_value"].copy()

**Handling Missing Values**

In [ ]:
# Considers only numeric columns as the imputer works on numeric data
housing_test_num = housing_test.drop("ocean_proximity", axis=1)

# Uses already fitted imputer to transforms the missing values in each column
housing_test_num_imputed = [CODE HERE FOR imputer TO TRANSFORM housing_test_num]

**Scaling Features**

In [ ]:
# Use already fitted scaler std_scaler to transform (or scale) dataset housing_test_num_imputed

housing_test_num_scaled = [CODE HERE]

In [ ]:
# Prints the shape of the transformed test set with only numerical attributes
housing_test_num_scaled.shape

**Encoding Categorical Attributes**

In [ ]:
# Sets a variable to attribute column
housing_test_cat = housing_test[["ocean_proximity"]]

# Uses already fitted categorical encoder cat_encoder to transform the categorical attributes housing_test_cat
# into one-hot-encoded columns
housing_test_cat_1hot = [CODE HERE]

**Combining Transformed Data**

In [ ]:
# Stacks numerically scaled data and categorically encoded data horizontally (column wise).
housing_test_transformed = np.hstack((housing_test_num_scaled, housing_test_cat_1hot))

# The above stacking can also be done over the following statement
# housing_test_transformed = np.append(housing_test_num_scaled, housing_test_cat_1hot, axis=1)

# Checks the shape of transformed train set
housing_test_transformed.shape

In [ ]:
# Just for reference, the test set is shown through DataFrame 
display(
    pd.DataFrame(
        housing_test_transformed, 
        index=housing_test_num.index,
        columns=list(housing_test_num.columns)+list(cat_encoder.categories_[0])
    )
)

In [63]:
# Saves the transformed test set to be used for machine learning training later
# NOTE: Target also gets saved in the last column

with open("./housing_test_transformed.npy", "wb") as f:
    np.save(
        f, 
        np.concatenate((housing_test_transformed, np.expand_dims(housing_test_labels, axis=1)), axis=1), 
        allow_pickle=False)

Now, appropriate machine learning model(s) can be trained over the transformed train set and can be tested against transformed test set to estimate model performance.